# 🤖 Bento 3D — Robô Mediador Social entre Idoso e Família (Brasil)

**Autores:** Matheus Henrique & Lucas Denner  
**Referência Científica:** Noguchi, Y., Kamide, H., & Tanaka, F. (2023). *How Should a Social Mediator Robot Convey Messages About the Self-Disclosures of Elderly People to Recipients?* International Journal of Social Robotics, 15, 1079–1099.

---
### 🚀 Como executar no Google Colab (GPU T4 Gratuita):
1. No menu superior do Colab, verifique se a GPU está ativa em **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.
2. Execute as células abaixo em ordem (ou clique em **Ambiente de execução → Executar tudo**).
3. Na última célula, será exibido um link público **`https://....trycloudflare.com`**. Clique nele para abrir o ambiente 3D com microfone e voz ativos!

In [ ]:
# @title 1. Clonar Repositório e Instalar Dependências (Node.js + Cloudflared)
import os

REPO_URL = "https://github.com/Umhenrique/robo-mediador-3d.git"
PROJECT_DIR = "/content/robo-mediador-3d"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

%cd {PROJECT_DIR}
!npm install

# Baixa Cloudflared para criar túnel HTTPS gratuito (necessário para permissão de microfone no navegador)
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

print("✅ Repositório e dependências configurados com sucesso!")

In [ ]:
# @title 2. Baixar KoboldCpp (CUDA) + LLM Local Qwen2.5-7B (Q4_K_M ~4.5GB VRAM) + Whisper Small STT
import os

os.makedirs("/content/models", exist_ok=True)

KOBOLD_BIN = "/content/koboldcpp-linux-x64-cuda1210"
LLM_MODEL = "/content/models/Qwen2.5-7B-Instruct-Q4_K_M.gguf"
WHISPER_MODEL = "/content/models/ggml-small-q5_1.bin"

if not os.path.exists(KOBOLD_BIN):
    print("⬇️ Baixando KoboldCpp com suporte a GPU CUDA...")
    !wget -q --show-progress https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64-cuda1210 -O {KOBOLD_BIN}
    !chmod +x {KOBOLD_BIN}

if not os.path.exists(LLM_MODEL):
    print("⬇️ Baixando LLM Qwen2.5-7B-Instruct-Q4_K_M.gguf (~4.5 GB)...")
    !wget -q --show-progress https://huggingface.co/bartowski/Qwen2.5-7B-Instruct-GGUF/resolve/main/Qwen2.5-7B-Instruct-Q4_K_M.gguf -O {LLM_MODEL}

if not os.path.exists(WHISPER_MODEL):
    print("⬇️ Baixando Whisper Small (Q5_1) para reconhecimento de voz em Português...")
    !wget -q --show-progress https://huggingface.co/ggerganov/whisper.cpp/resolve/main/ggml-small-q5_1.bin -O {WHISPER_MODEL}

print("✅ Modelos carregados no disco do Colab!")

In [ ]:
# @title 3. Iniciar LLM na GPU + Servidor Bento 3D + Gerar Link HTTPS Público
import subprocess
import time
import re
import urllib.request

# 1. Inicia o KoboldCpp em background na porta 5001
print("🧠 Iniciando KoboldCpp (Qwen2.5-7B + Whisper Small) na GPU...")
kobold_log = open("/content/kobold.log", "w")
kobold_proc = subprocess.Popen([
    "/content/koboldcpp-linux-x64-cuda1210",
    "--model", "/content/models/Qwen2.5-7B-Instruct-Q4_K_M.gguf",
    "--whispermodel", "/content/models/ggml-small-q5_1.bin",
    "--contextsize", "4096",
    "--gpulayers", "99",
    "--usecublas",
    "--flashattention",
    "--quantkv", "1",
    "--port", "5001",
    "--skiplauncher"
], stdout=kobold_log, stderr=subprocess.STDOUT)

# Aguarda API do KoboldCpp ficar online
for i in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:5001/api/v1/model", timeout=2) as r:
            if r.status == 200:
                print("✅ LLM Local + Whisper prontos na porta 5001!")
                break
    except Exception:
        time.sleep(2)
else:
    print("⚠️ Aviso: KoboldCpp ainda está carregando. Verifique /content/kobold.log se necessário.")

# 2. Inicia o servidor Node.js do Bento 3D na porta 3030
print("🤖 Iniciando servidor Bento 3D na porta 3030...")
node_log = open("/content/bento.log", "w")
node_proc = subprocess.Popen(
    ["node", "src/server.js"],
    cwd="/content/robo-mediador-3d",
    stdout=node_log,
    stderr=subprocess.STDOUT
)
time.sleep(2)

# 3. Abre o túnel HTTPS com Cloudflared
print("🌐 Gerando link HTTPS público (Cloudflare Tunnel)...")
tunnel_log = open("/content/tunnel.log", "w+")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:3030"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

public_url = None
for _ in range(30):
    time.sleep(1)
    tunnel_log.seek(0)
    content = tunnel_log.read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
    if m:
        public_url = m.group(0)
        break

print("\n" + "="*75)
if public_url:
    print(f"🎉 BENTO 3D ONLINE! CLIQUE NO LINK ABAIXO PARA ABRIR:")
    print(f"👉 {public_url}")
else:
    print("⚠️ Não foi possível capturar a URL automaticamente. Veja /content/tunnel.log")
print("="*75 + "\n")

# Mantém a célula viva enquanto o servidor estiver rodando
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("🛑 Encerrando servidores...")
    tunnel_proc.terminate()
    node_proc.terminate()
    kobold_proc.terminate()